# SVCM — ITI-97 — Expansion ValueSet & Lookup CodeSystem

**Profil** : IHE ITI SVCM
**Acteur testé** : SVCM-Terminology Consumer
**Transaction** : SVCM-ITI-97
**Référence Gazelle** : https://interop.esante.gouv.fr/tm/testing/testsDefinition/viewTestPage.seam?id=12208&cid=48550

Deux scénarios :
1. `$expand` sur un ValueSet SNOMED (structures anatomiques, code `302553009`)
2. `$lookup` POST + `$validate-code` + `$subsumes` sur CIM-11 (code `04`)

## Configuration

In [ ]:
import requests
import json
import time
from urllib.parse import quote

FHIR_BASE = "https://smt.esante.gouv.fr/fhir"
API_KEY   = "ZiaoDxF8Je0CfBNzu..."   # Remplacer par la clé API complète

HTTP_RETRIES = 3
HTTP_BACKOFF = 2

HEADERS_JSON = {
    "Accept": "application/fhir+json",
    "User-Agent": "SVCM-Consumer-CAD-CDM",
}
HEADERS_POST = {
    "Accept": "application/fhir+json",
    "Content-Type": "application/fhir+json",
    "User-Agent": "SVCM-Consumer-CAD-CDM",
}

def http_get(url, headers=None):
    if headers is None:
        headers = HEADERS_JSON
    for attempt in range(1, HTTP_RETRIES + 1):
        try:
            print(f"  → GET {url}")
            r = requests.get(url, headers=headers, allow_redirects=True)
            if r.status_code == 200:
                return r
            raise Exception(f"HTTP {r.status_code}: {r.text[:200]}")
        except Exception as e:
            print(f"  ✗ tentative {attempt}/{HTTP_RETRIES} : {e}")
            if attempt == HTTP_RETRIES:
                raise
            time.sleep(HTTP_BACKOFF ** attempt)

def http_post(url, body=None, headers=None):
    if headers is None:
        headers = HEADERS_POST
    for attempt in range(1, HTTP_RETRIES + 1):
        try:
            print(f"  → POST {url}")
            r = requests.post(url, json=body, headers=headers, allow_redirects=True)
            if r.status_code == 200:
                return r
            raise Exception(f"HTTP {r.status_code}: {r.text[:200]}")
        except Exception as e:
            print(f"  ✗ tentative {attempt}/{HTTP_RETRIES} : {e}")
            if attempt == HTTP_RETRIES:
                raise
            time.sleep(HTTP_BACKOFF ** attempt)

def print_keys(data, *keys):
    subset = {k: data.get(k) for k in keys if k in data}
    print(json.dumps(subset, indent=2, ensure_ascii=False))

print("Configuration OK — prêt à exécuter les étapes.")


---
## Étapes 0–30 — Expansion d'un ValueSet SNOMED (`$expand`)

**Requête** : `GET /fhir/ValueSet/jdv-localisation-anatomique-cisis/$expand?code=302553009&_format=json`
**Objectif** : Récupérer les codes de localisation anatomique (sous-hiérarchie SNOMED `302553009`).

In [ ]:
# Étape 0  — Construction de la requête
# Étape 10 — TRANSACTION : GET $expand
url_expand = f"{FHIR_BASE}/ValueSet/jdv-localisation-anatomique-cisis/$expand?code=302553009&_format=json"

r_expand = http_get(url_expand)
vs_expanded = r_expand.json()

# Étape 20 — Réception et intégration
expansion = vs_expanded.get("expansion", {})
contains  = expansion.get("contains", [])
total     = expansion.get("total", len(contains))

print(f"[PREUVE étape 30] ValueSet expansé :")
print_keys(vs_expanded, "resourceType", "id", "url", "version", "status")
print(f"\nExpansion total : {total}")
print(f"Concepts retournés : {len(contains)}")
if contains:
    print("\nPremiers concepts :")
    for c in contains[:10]:
        print(f"  {c.get('system')} | {c.get('code')} — {c.get('display')}")
    if len(contains) > 10:
        print(f"  ... ({len(contains) - 10} autres)")


---
## Étapes 40–70 — Lookup, validate-code et subsumes sur CIM-11

**Requêtes** :
- `POST /fhir/CodeSystem/$lookup` — lookup du code `04` (maladies du système immunitaire) en CIM-11
- `GET /fhir/CodeSystem/$validate-code` — validation du code `04`
- `GET /fhir/CodeSystem/$subsumes` — relation de subsomption entre `04` et des sous-codes

In [ ]:
# Étape 40 — Construction de la requête
# Étape 50 — TRANSACTION : POST $lookup code 04 en CIM-11
CIM11_SYSTEM = "https://smt.esante.gouv.fr/terminologie-cim11-mms"

lookup_body = {
    "resourceType": "Parameters",
    "parameter": [
        {"name": "system", "valueUri": CIM11_SYSTEM},
        {"name": "code",   "valueCode": "04"},
    ],
}
r_lookup = http_post(f"{FHIR_BASE}/CodeSystem/$lookup?_format=json", lookup_body)
lookup_result = r_lookup.json()

# Étape 60 — Réception et intégration
params = {p["name"]: p for p in lookup_result.get("parameter", [])}
display_val = params.get("display", {}).get("valueString", "")
name_val    = params.get("name",    {}).get("valueString", "")
print(f"[PREUVE étape 60] $lookup code 04 :")
print(f"  display : {display_val}")
print(f"  name    : {name_val}")
print()


In [ ]:
# $validate-code — code 04 dans CIM-11 (GET)
url_vc_get = (
    f"{FHIR_BASE}/CodeSystem/$validate-code"
    f"?url={quote(CIM11_SYSTEM)}&code=04&_format=json"
)
r_vc = http_get(url_vc_get)
vc_result = r_vc.json()
vc_params  = {p["name"]: p for p in vc_result.get("parameter", [])}
print(f"$validate-code (GET) — result : {vc_params.get('result', {}).get('valueBoolean')}")
print(f"  display : {vc_params.get('display', {}).get('valueString')}")

# $subsumes — 04 vs 3B62.0Y  (attendu : not-subsumed — chapitres différents)
url_sub1 = (
    f"{FHIR_BASE}/CodeSystem/$subsumes"
    f"?system={quote(CIM11_SYSTEM)}&codeA=04&codeB=3B62.0Y&_format=json"
)
r_sub1 = http_get(url_sub1)
sub1 = {p["name"]: p for p in r_sub1.json().get("parameter", [])}
print(f"\n$subsumes 04 vs 3B62.0Y : {sub1.get('outcome', {}).get('valueCode')}  (attendu: not-subsumed)")

# $subsumes — 04 vs 4A01.10  (attendu : subsumes)
url_sub2 = (
    f"{FHIR_BASE}/CodeSystem/$subsumes"
    f"?system={quote(CIM11_SYSTEM)}&codeA=04&codeB=4A01.10&_format=json"
)
r_sub2 = http_get(url_sub2)
sub2 = {p["name"]: p for p in r_sub2.json().get("parameter", [])}
print(f"$subsumes 04 vs 4A01.10  : {sub2.get('outcome', {}).get('valueCode')}  (attendu: subsumes)")
